# Train the BERT gap model on Colab (GPU)

Prereq: `colab_bundle.zip` (made by `scripts/package_for_colab.sh`) uploaded to `MyDrive/disfluency/`.
Runtime: *Runtime > Change runtime type > T4 GPU*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/disfluency/ckpt', exist_ok=True)

In [ ]:
!rm -rf /content/proj && mkdir -p /content/proj
!cp /content/drive/MyDrive/disfluency/colab_bundle.zip /content/
!cd /content && unzip -q -o colab_bundle.zip -d proj && ls proj proj/data/gaps
!pip -q install transformers scikit-learn pyyaml

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Main run (phones + simulated recognizer noise)

In [ ]:
!cd /content/proj && python -I scripts/train.py --epochs 3 --batch 64 --device cuda --save \
    --out /content/drive/MyDrive/disfluency/ckpt/main

## Ablations (run one at a time; each writes its own folder on Drive)

In [ ]:
# text-only BERT (no phones)
# !cd /content/proj && python -I scripts/train.py --epochs 3 --batch 64 --device cuda --no-phones --out /content/drive/MyDrive/disfluency/ckpt/no_phones

# clean phones (no recognizer-noise simulation)
# !cd /content/proj && python -I scripts/train.py --epochs 3 --batch 64 --device cuda --no-noise --out /content/drive/MyDrive/disfluency/ckpt/no_noise

In [ ]:
import json
m = json.load(open('/content/drive/MyDrive/disfluency/ckpt/main/dev_metrics.json'))[-1]
print({k: m[k] for k in ['unit_type_acc', 'unit_type_macro_f1', 'restoration_acc', 'restoration_acc_known_targets']})
for t, v in m['per_class'].items():
    print(f"{t:20s} P={v['p']:.3f} R={v['r']:.3f} F1={v['f1']:.3f} n={v['support']}")